# Milestone 3 Baseline Demonstration

**Group 58 — CSCI 3052U**

This notebook reproduces our data-cleaning pipeline, majority-class benchmark, and character TF–IDF Logistic Regression baseline.

Languages: Japanese, Tamil, Portuguese, Turkish, Chinese and Persian.

Chinese examples are complete user histories. Persian labels are provisional
annotation-presence labels, not clinical diagnoses. Japanese labels are automatically assigned using a depression lexicon. Tamil examples are automatic speech transcripts with recording-level labels; speaker-independent splitting cannot be verified.

Only validation results are reported. Reserved test sets remain unevaluated.
The age-intervention experiment has not yet been performed.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

print("Colab notebook Python:", sys.version)

REPO = Path("/content/age_bias_milestone3")
ENV = Path("/content/age_bias_m3_python311")
COMMIT = "568b1ec9b4c15ac01378e2c4f4c5a70e13478587"

def run(command):
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    result.check_returncode()

# Install the environment-management tool.
run([sys.executable, "-m", "pip", "install", "uv"])
UV = [sys.executable, "-m", "uv"]

# Download the project or refresh the existing clone.
if not REPO.exists():
    run([
        "git", "clone",
        "https://github.com/m3hreen/machine_learning_project.git",
        str(REPO),
    ])
else:
    run(["git", "-C", str(REPO), "fetch", "origin"])

# Use the exact six-language code checkpoint.
run([
    "git", "-C", str(REPO),
    "checkout", "--detach", COMMIT,
])

# Create an isolated Python 3.11 environment.
if not (ENV / "bin/python").exists():
    run(UV + ["venv", "--python", "3.11", str(ENV)])

PYTHON = str(ENV / "bin/python")

# Install the recorded project dependencies.
run(UV + [
    "pip", "install",
    "--python", PYTHON,
    "-r", str(REPO / "requirements-lock.txt"),
])

run(UV + ["pip", "check", "--python", PYTHON])

os.chdir(REPO)

print("\nPython used for the baseline scripts:")
run([PYTHON, "--version"])

print("Original Mac run used Python 3.11.17.")
print("Project:", REPO)
print("Baseline checkpoint:", COMMIT)
print("\nSetup complete. Continue to Cell 3.")

Colab notebook Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 46.4 MB/s eta 0:00:00

Cloning into '/content/age_bias_milestone3'...

HEAD is now at 568b1ec Add Japanese cleaning pipeline and six-language validation baselines

Using CPython 3.11.17
Creating virtual environment at: age_bias_m3_python311
Activate with: source age_bias_m3_python311/bin/activate

Using Python 3.11.17 environment at: age_bias_m3_python311
Resolved 13 packages in 525ms
Prepared 13 packages in 14.26s
Installed 13 packages in 71ms
 + joblib==1.4.2
 + numpy==1.26.4
 + packaging==26.3
 + pandas==2.2.3
 + pyarrow==17.0.0
 + pygame==2.5.1
 + python-dateutil==2.9.0.post0
 + pytz==2026.5
 + scikit-learn==1.5.2
 + scipy==1.13.1
 + six==1.17.0
 + threadpoolctl==3.7.0
 + tzdata==2026.5

Using Python 3.11.17 environment at: age_bias_m3_python311
Checked 13 packages in 0.48ms
All installed packages are compatible


Python used for the baseline scripts:
P

Upload the following **seven files covering six languages** from your computer:

| Language | Required file |
|---|---|
| Chinese | `CNSD_dataset.json` |
| Persian | `result.csv` |
| Portuguese | `dataset_ideacao (3).csv` |
| Turkish | `train-00000-of-00001.parquet` |
| Turkish | `test-00000-of-00001.parquet` |
| Tamil | `tamil_audio_dataset.csv` |
| Japanese | `twitter_Japanese.csv` |


In [2]:
from google.colab import files
import csv
import io
import json
import subprocess
import tempfile
from pathlib import Path

REQUIRED = {
    "Chinese": "data/raw/chinese/CNSD_dataset.json",
    "Persian": "data/raw/persian/result.csv",
    "Portuguese": "data/raw/portuguese/dataset_ideacao (3).csv",
    "Turkish train": "data/raw/turkish/train-00000-of-00001.parquet",
    "Turkish test": "data/raw/turkish/test-00000-of-00001.parquet",
    "Tamil": "data/raw/tamil/tamil_audio_dataset.csv",
    "Japanese": "data/raw/japanese/twitter_Japanese.csv",
}

def identify(content):
    # Inspect Parquet using the project's Python environment.
    if content[:4] == b"PAR1":
        with tempfile.TemporaryDirectory() as folder:
            temporary = Path(folder) / "check.parquet"
            temporary.write_bytes(content)

            result = subprocess.run(
                [
                    PYTHON, "-c",
                    """
import json
import pyarrow.parquet as pq
import sys

metadata = pq.ParquetFile(sys.argv[1])
print(json.dumps({
    "columns": metadata.schema_arrow.names,
    "rows": metadata.metadata.num_rows
}))
""",
                    str(temporary),
                ],
                capture_output=True,
                text=True,
                check=True,
            )

        metadata = json.loads(result.stdout)

        if set(metadata["columns"]) == {"text", "labels"}:
            return {
                40317: "Turkish train",
                17280: "Turkish test",
            }.get(metadata["rows"])

        return None

    try:
        text = content.decode("utf-8-sig")
    except UnicodeDecodeError:
        return None

    # Chinese user histories.
    try:
        document = json.loads(text)
    except json.JSONDecodeError:
        document = None

    if document is not None:
        if (
            isinstance(document, dict)
            and document
            and all(
                isinstance(item, dict)
                and {"inpput", "output"}.issubset(item)
                for item in document.values()
            )
        ):
            return "Chinese"
        return None

    # Recognize the released CSV schemas.
    try:
        dialect = csv.Sniffer().sniff(
            text[:20000], delimiters=",;\t"
        )
        reader = csv.DictReader(io.StringIO(text), dialect=dialect)
        headers = set(reader.fieldnames or [])
        rows = list(reader)
    except (csv.Error, TypeError):
        return None

    if not rows:
        return None

    if {"text", "target"}.issubset(headers):
        return "Portuguese"

    if (
        "prompt" in headers
        and any(header.startswith("mental_state") for header in headers)
    ):
        return "Persian"

    if {"audio_text", "label"}.issubset(headers):
        return "Tamil"

    if {"tweet", "label"}.issubset(headers):
        # Schema alone cannot distinguish Japanese from Korean/English.
        # Require Japanese kana in most nonblank tweets.
        import re

        texts = [(row.get("tweet") or "").strip() for row in rows]
        texts = [text for text in texts if text]
        kana_count = sum(
            bool(re.search(r"[\u3040-\u30ff]", text))
            for text in texts
        )
        if texts and kana_count / len(texts) > 0.5:
            return "Japanese"

    return None


# Upload in batches until all seven files have been supplied.
# Each run requires explicit uploads, avoiding stale runtime files.
received = set()

while received != set(REQUIRED):
    print("\nStill needed:", ", ".join(sorted(set(REQUIRED) - received)))
    print("Select one or more files. Filenames may differ.")

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(
            "Upload cancelled. Rerun this cell when the files are ready."
        )

    for name, content in uploaded.items():
        language = identify(content)

        if language is None:
            print(f"Skipped {name!r}: unrecognized content.")
            continue

        destination = REPO / REQUIRED[language]

        if language in received:
            if destination.read_bytes() != content:
                raise ValueError(
                    f"Different files both match {language}; "
                    "check which source file is correct."
                )
            print(f"Already received: {language}")
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
        received.add(language)
        print(f"Saved: {name} → {REQUIRED[language]}")

    del uploaded

print("\nAll seven required files are available for six languages.")


Still needed: Chinese, Japanese, Persian, Portuguese, Tamil, Turkish test, Turkish train
Select one or more files. Filenames may differ.


Saving CNSD_dataset.json to CNSD_dataset.json
Saving dataset_ideacao (3).csv to dataset_ideacao (3).csv
Saving result.csv to result.csv
Saving tamil_audio_dataset copy.csv to tamil_audio_dataset copy.csv
Saving test-00000-of-00001.parquet to test-00000-of-00001.parquet
Saving train-00000-of-00001.parquet to train-00000-of-00001.parquet
Saving twitter_Japanese copy.csv to twitter_Japanese copy.csv
Saved: CNSD_dataset.json → data/raw/chinese/CNSD_dataset.json
Saved: dataset_ideacao (3).csv → data/raw/portuguese/dataset_ideacao (3).csv
Saved: result.csv → data/raw/persian/result.csv
Saved: tamil_audio_dataset copy.csv → data/raw/tamil/tamil_audio_dataset.csv
Saved: test-00000-of-00001.parquet → data/raw/turkish/test-00000-of-00001.parquet
Saved: train-00000-of-00001.parquet → data/raw/turkish/train-00000-of-00001.parquet
Saved: twitter_Japanese copy.csv → data/raw/japanese/twitter_Japanese.csv

All seven required files are available for six languages.


In [3]:
scripts = [
    "src/prepare_tabular.py",
    "src/prepare_chinese.py",
    "src/prepare_persian.py",
    "src/prepare_tamil.py",
    "src/prepare_japanese.py",
    "src/run_baselines.py",
]

for script in scripts:
    print(f"\n{'=' * 60}\nRUNNING: {script}", flush=True)
    run([PYTHON, str(REPO / script)])

print("\nSix-language pipeline completed.")
print("Test sets remain unevaluated.")


RUNNING: src/prepare_tabular.py
{
  "portuguese": {
    "raw_rows": 3788,
    "blank_rows_excluded": 0,
    "conflicting_unique_texts": 8,
    "conflicting_rows_excluded": 16,
    "duplicate_rows_excluded_after_conflicts": 768,
    "remaining_rows": 3004,
    "remaining_label_counts": {
      "0": 2141,
      "1": 863
    },
    "source_sha256": {
      "dataset_ideacao (3).csv": "07cc1bb671238fdc533b47cbc1efbc4046029af4ed02c32137fc828845bfa307"
    }
  },
  "turkish": {
    "raw_rows": 57597,
    "blank_rows_excluded": 10,
    "conflicting_unique_texts": 5,
    "conflicting_rows_excluded": 11,
    "duplicate_rows_excluded_after_conflicts": 2182,
    "remaining_rows": 55394,
    "remaining_label_counts": {
      "0": 27847,
      "1": 27547
    },
    "original_shared_unique_texts": 572,
    "remaining_source_split_counts": {
      "train": 39035,
      "test": 16359
    },
    "source_sha256": {
      "train-00000-of-00001.parquet": "1a3f85151e5589ebde07cc69fb6f981396ecdbb48f8170ab86

In [4]:
import csv
import math
from collections import Counter
from IPython.display import Markdown, display

with (REPO / "reports/baseline_validation.csv").open(
    encoding="utf-8", newline=""
) as file:
    results = list(csv.DictReader(file))

expected_f1 = {
    ("portuguese", "majority_class"): 0.4159,
    ("portuguese", "char_tfidf_logistic_regression"): 0.7688,
    ("turkish", "majority_class"): 0.3354,
    ("turkish", "char_tfidf_logistic_regression"): 0.9632,
    ("chinese", "majority_class"): 0.3286,
    ("chinese", "char_tfidf_logistic_regression"): 0.7642,
    ("persian", "majority_class"): 0.4804,
    ("persian", "char_tfidf_logistic_regression"): 0.4804,
    ("tamil", "majority_class"): 0.3902,
    ("tamil", "char_tfidf_logistic_regression"): 0.9869,
    ("japanese", "majority_class"): 0.3367,
    ("japanese", "char_tfidf_logistic_regression"): 0.8468,
}

model_names = {
    "majority_class": "Majority class",
    "char_tfidf_logistic_regression": "TF–IDF + Logistic Regression",
}

# Verify that every expected result appears exactly once.
keys = [(row["language"], row["model"]) for row in results]
counts = Counter(keys)

missing = set(expected_f1) - set(keys)
unexpected = set(keys) - set(expected_f1)
duplicates = [key for key, count in counts.items() if count > 1]

if missing or unexpected or duplicates:
    raise ValueError(
        f"Missing results: {sorted(missing)}\n"
        f"Unexpected results: {sorted(unexpected)}\n"
        f"Duplicate results: {sorted(duplicates)}"
    )

if any(row["partition"] != "validation" for row in results):
    raise ValueError("Expected validation results only.")

table = [
    "| Language | Model | Accuracy | Balanced accuracy | Macro-F1 | Positive recall |",
    "|---|---|---:|---:|---:|---:|",
]

differences = []

for row in results:
    key = (row["language"], row["model"])
    metrics = {
        name: float(row[name])
        for name in [
            "accuracy", "balanced_accuracy",
            "macro_f1", "positive_recall",
        ]
    }

    if not all(
        math.isfinite(value) and 0 <= value <= 1
        for value in metrics.values()
    ):
        raise ValueError(f"Invalid metrics for {key}: {metrics}")

    table.append(
        f"| {row['language']} | {model_names[row['model']]} | "
        f"{metrics['accuracy']:.4f} | "
        f"{metrics['balanced_accuracy']:.4f} | "
        f"{metrics['macro_f1']:.4f} | "
        f"{metrics['positive_recall']:.4f} |"
    )

    observed = round(metrics["macro_f1"], 4)
    if observed != expected_f1[key]:
        differences.append((key, expected_f1[key], observed))

display(Markdown("\n".join(table)))

if differences:
    print("Macro-F1 differences from the Mac run:")
    for key, expected, observed in differences:
        print(f"{key}: expected {expected:.4f}, observed {observed:.4f}")
    print("Investigate source hashes, code revision, and environment versions.")
else:
    print("All 12 macro-F1 values match the Mac run to four decimals.")

print("This check compares macro-F1 only; runtimes may differ.")
print("Reserved test sets remain unevaluated.")

| Language | Model | Accuracy | Balanced accuracy | Macro-F1 | Positive recall |
|---|---|---:|---:|---:|---:|
| portuguese | Majority class | 0.7121 | 0.5000 | 0.4159 | 0.0000 |
| portuguese | TF–IDF + Logistic Regression | 0.8419 | 0.7358 | 0.7688 | 0.4855 |
| turkish | Majority class | 0.5047 | 0.5000 | 0.3354 | 0.0000 |
| turkish | TF–IDF + Logistic Regression | 0.9632 | 0.9632 | 0.9632 | 0.9550 |
| chinese | Majority class | 0.4894 | 0.5000 | 0.3286 | 0.0000 |
| chinese | TF–IDF + Logistic Regression | 0.7660 | 0.7681 | 0.7642 | 0.6667 |
| persian | Majority class | 0.9247 | 0.5000 | 0.4804 | 0.0000 |
| persian | TF–IDF + Logistic Regression | 0.9247 | 0.5000 | 0.4804 | 0.0000 |
| tamil | Majority class | 0.6400 | 0.5000 | 0.3902 | 0.0000 |
| tamil | TF–IDF + Logistic Regression | 0.9880 | 0.9833 | 0.9869 | 0.9667 |
| japanese | Majority class | 0.5076 | 0.5000 | 0.3367 | 0.0000 |
| japanese | TF–IDF + Logistic Regression | 0.8477 | 0.8468 | 0.8468 | 0.7835 |

All 12 macro-F1 values match the Mac run to four decimals.
This check compares macro-F1 only; runtimes may differ.
Reserved test sets remain unevaluated.


## Findings and discrepancy analysis

- Portuguese Logistic Regression improves on the majority benchmark, but its positive recall is only 48.55%, missing more than half of validation positives.
- Turkish achieves strong validation performance. Exact-text overlap was removed, but near-duplicates, author overlap and collection artifacts still need review.
- Chinese results apply to complete user histories. Only 47 validation users were evaluated, so the estimate is sensitive to a few errors.
- Persian Logistic Regression predicts only the comparison class. Its 92.47% accuracy hides zero positive recall. Labels are provisional and require review.
- Tamil achieves 0.9869 validation macro-F1 on automatic speech transcripts. Repeated sentences and possible speaker overlap may inflate performance; speaker-independent splitting cannot be verified.
- Japanese achieves 0.8468 validation macro-F1 and 78.35% positive recall. Its lexicon-derived labels may reward learning the labeling keywords rather than broader symptom evidence.
- These results use full cleaned datasets rather than the planned reviewed 50-positive/50-comparison age-study samples.
- This is a course-level baseline implementation, not an exact reproduction of published neural models. Their scores use different evaluation protocols.
- The notebook compares all 12 validation macro-F1 values against the recorded Mac run. Any differences require investigation of source hashes, code revision and dependency versions.
- No age effects or final test results have been measured.

## Start / Stop / Continue

**Start:** Review Persian labels; compare balanced class weights using validation data; investigate Turkish collection artifacts, Tamil speaker overlap and Japanese keyword dependence; review candidate texts for age-edit suitability.

**Stop:** Relying on accuracy alone, treating comparison labels as clinically healthy, interpreting high validation scores as proof of clinical validity, or ranking languages as if their tasks and units were identical.

**Continue:** Preserve raw data, exclude exact duplicates and conflicting-label groups, keep known authors or speakers together where identifiers are available, fit TF–IDF only on training data, record configurations and keep test sets reserved.